# 手势识别（机械臂主机直接运行）

在 Jupyter 内核里直接打开摄像头 → MediaPipe 识别 → 实时回显 → 输出 `GestureCommand`。

## 运行前

1. **先把 `Face.ipynb` 的内核 Shutdown**。它的 `camera_thread` 会在后台一直占用
   `/dev/video0`，本 notebook 只能打开设备但读不到画面（表现为卡住不动）。
2. 依赖已在同目录 `pylibs/`（mediapipe **0.10.18** + protobuf 4.25.3 + flatbuffers/absl/sounddevice），
   本 notebook 自动加入 `sys.path`，**不需要也不能** `pip install`（板子没有外网 DNS）。

   > 注意：`mediapipe 1.x`（manylinux_2_28）在本板 CPU 上会因 `rcpc` 指令报
   > `FATAL ERROR ... compiled with rcpc` 并直接杀进程（表现为内核重启、`NameError`），
   > 必须用 0.10.x。旧的 1.2.3 已移到 `pylibs/_broken_mediapipe_1.2.3`。

## 顺序

环境准备 → 模型校验 → 摄像头自检 → 实时识别 → 结果汇总

每个单元都会在开头检查前置状态，失败时给出中文原因，不会出现 `NameError`。

In [ ]:
# 1) 环境准备（唯一初始化点）——本单元必须成功，后面单元依赖它
import os, sys, traceback, importlib
from pathlib import Path


def find_project():
    """向上查找含 gesture/recognizer.py 的目录。"""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "gesture" / "recognizer.py").is_file():
            return base
    raise SystemExit("未找到 gesture/ 目录，请确认 gesture、voice、pylibs 与 notebook 在同一目录")


PROJECT_SRC = find_project()
PYLIBS = PROJECT_SRC / "pylibs"
# pylibs 优先后插入；若存在 pylibs010（新版安装目录）则优先级最高
for p in (PROJECT_SRC / "pylibs", PROJECT_SRC / "pylibs010"):
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("python :", sys.version.split()[0])
print("工程目录:", PROJECT_SRC)
print("pylibs :", PYLIBS, "存在" if PYLIBS.is_dir() else "不存在")

GESTURE_READY = False
try:
    import cv2, numpy, mediapipe
    print("cv2      :", cv2.__version__)
    print("numpy    :", numpy.__version__)
    print("mediapipe:", mediapipe.__version__, "|", mediapipe.__file__)
    if mediapipe.__version__.startswith("1."):
        raise SystemExit(
            "mediapipe %s 与本板 CPU 不兼容：它用到了 rcpc 指令，创建识别器时会直接杀掉进程"
            "（现象是内核重启、之后报 NameError）。\n"
            "本板必须用 mediapipe 0.10.x；正确的包在 pylibs/mediapipe，"
            "旧的 1.2.3 已移到 pylibs/_broken_mediapipe_1.2.3。" % mediapipe.__version__)
    from gesture.gesture_command import GESTURE_CODES
    from gesture.recognizer import HandRecognizer
    from gesture.command_gate import CommandGate
    from gesture.logger import RecognitionLogger
    GESTURE_READY = True
    print("\n依赖与 gesture 模块导入成功")
    print("手势映射:", GESTURE_CODES)
except Exception:
    print("\n导入失败，完整堆栈如下：")
    traceback.print_exc()
    print("\n修复提示：若 mediapipe 相关报错，确认 pylibs/mediapipe 是否完整；"
          "若 gesture.* 报错，确认 gesture 目录是否上传齐全。")

In [ ]:
# 2) 模型校验
if not globals().get("GESTURE_READY", False):
    raise SystemExit("请先成功运行单元 1（环境准备）")

MODEL_DIR = PROJECT_SRC / "gesture" / "models"
models = sorted(p.name for p in MODEL_DIR.glob("*.task"))
print("模型目录:", MODEL_DIR)
print("模型文件:", models)
missing = {"gesture_recognizer.task", "hand_landmarker.task"} - set(models)
if missing:
    raise SystemExit("模型缺失: %s，请重新上传 gesture/models/ 目录" % sorted(missing))
print("模型校验通过")

In [ ]:
# 3) 摄像头自检（后台线程读帧，避免 read() 卡死整个单元）
if not globals().get("GESTURE_READY", False):
    raise SystemExit("请先成功运行单元 1（环境准备）")

import cv2, threading, time
from IPython.display import display, Image


class CameraReader:
    """后台线程持续读帧，主线程只取最新一帧。设备被占用时不会阻塞主线程。"""

    def __init__(self, index=0, width=320, height=240, fps=30):
        self.cap = cv2.VideoCapture(index)
        if self.cap.isOpened():
            self.cap.set(3, width)
            self.cap.set(4, height)
            self.cap.set(5, fps)
        self._frame = None
        self._lock = threading.Lock()
        self._stop = False
        self._t = None

    def start(self):
        if not self.cap.isOpened():
            return False
        self._t = threading.Thread(target=self._loop, daemon=True)
        self._t.start()
        return True

    def _loop(self):
        while not self._stop:
            ok, frame = self.cap.read()
            if ok and frame is not None:
                with self._lock:
                    self._frame = frame
            else:
                time.sleep(0.05)

    def latest(self):
        with self._lock:
            return None if self._frame is None else self._frame.copy()

    def wait_first(self, timeout=8.0):
        """等待首帧；超时说明设备被其它进程占用。"""
        end = time.time() + timeout
        while time.time() < end:
            f = self.latest()
            if f is not None:
                return f
            time.sleep(0.1)
        return None

    def close(self):
        self._stop = True
        if self._t is not None:
            self._t.join(timeout=1.0)
        self.cap.release()


cam = CameraReader(0)
if not cam.start():
    raise SystemExit("无法打开 /dev/video0：请检查接线，或先停止占用摄像头的内核（如 Face.ipynb）")

frame = cam.wait_first(8.0)
if frame is None:
    cam.close()
    raise SystemExit(
        "摄像头能打开但 8 秒内读不到画面：\n"
        "  1) 最常见原因：另一个内核（Face.ipynb）的 camera_thread 仍占用设备 → Kernel → Shutdown 后重试\n"
        "  2) 终端确认：ls -l /dev/video*  以及  fuser -v /dev/video0\n"
        "  3) 若编号不是 0，把上面的 CameraReader(0) 改成 CameraReader(1) 等"
    )

ok_jpg, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
if ok_jpg:
    display(Image(data=buf.tobytes(), width=480))
print("画面尺寸:", frame.shape, "| 平均亮度:", round(float(frame.mean()), 1))
cam.close()
print("摄像头自检通过")

In [ ]:
# 4) 实时手势识别（20 秒，画面实时回显）
if not globals().get("GESTURE_READY", False):
    raise SystemExit("请先成功运行单元 1（环境准备）")
if "CameraReader" not in globals():
    raise SystemExit("请先成功运行单元 3（摄像头自检）")

import cv2, json, time
import ipywidgets as widgets
from IPython.display import display

OUT_FILE = PROJECT_SRC / "gesture_commands.jsonl"
LOG_FILE = PROJECT_SRC / "recognition_log.csv"
RUN_SECONDS = 20

print("正在加载 MediaPipe 模型 ...", flush=True)
recognizer = HandRecognizer(min_confidence=0.5, smooth_alpha=0.6)
print("模型加载完成", flush=True)

gate = CommandGate(min_confidence=0.5, use_calibration=False)
logger = RecognitionLogger(str(LOG_FILE))
preview = widgets.Image(format="jpeg", width=480, height=360)
display(preview)

cam = CameraReader(0)
if not cam.start():
    raise SystemExit("无法打开 /dev/video0")
if cam.wait_first(8.0) is None:
    cam.close()
    raise SystemExit("读不到画面：请先 Shutdown 占用摄像头的内核（如 Face.ipynb）后重试")

records = []
start = time.time()
last_frame_at = time.time()
try:
    while time.time() - start < RUN_SECONDS:
        frame = cam.latest()
        if frame is None:
            if time.time() - last_frame_at > 5:
                print("超过 5 秒无画面，退出")
                break
            time.sleep(0.05)
            continue
        last_frame_at = time.time()

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = recognizer.process_frame(rgb)
        gid = res.gesture_id if res is not None else "NONE"
        conf = res.confidence if res is not None else 0.0
        raw_conf = res.base_conf if res is not None else 0.0

        cmd = gate.update(gid, conf, time.time())
        logger.log(gid, conf, cmd is not None, raw_confidence=raw_conf)

        color = (0, 200, 0) if cmd is not None else (200, 200, 200)
        cv2.putText(frame, "%s %.2f" % (gid, conf), (8, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        if cmd is not None:
            print(">> 命令:", cmd.to_json(), flush=True)
            payload = json.loads(cmd.to_json())
            records.append(payload)
            with open(OUT_FILE, "a", encoding="utf-8") as f:
                f.write(cmd.to_json() + "\n")

        ok_jpg, jpg = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 60])
        if ok_jpg:
            preview.value = jpg.tobytes()
        time.sleep(0.02)
finally:
    cam.close()
    recognizer.close()

print("本次运行 %ds，下发 %d 条手势命令" % (RUN_SECONDS, len(records)))
for r in records[-10:]:
    print(r["gesture_id"], "code=", r["gesture_code"], "conf=", r["confidence"])

In [ ]:
# 5) 结果汇总（可独立运行）
import json
from collections import Counter
from pathlib import Path

base = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "gesture").is_dir()), None)
OUT_FILE = base / "gesture_commands.jsonl"
LOG_FILE = base / "recognition_log.csv"

rows = []
if OUT_FILE.exists():
    for ln in OUT_FILE.read_text(encoding="utf-8").splitlines():
        ln = ln.strip()
        if ln.startswith("{"):
            rows.append(json.loads(ln))

print("命令文件:", OUT_FILE, "存在" if OUT_FILE.exists() else "不存在")
print("累计命令数:", len(rows))
print("分布:", Counter(r["gesture_id"] for r in rows))
if LOG_FILE.exists():
    print("识别日志行数:", len(LOG_FILE.read_text(encoding="utf-8").splitlines()))
rows[-5:]

## 与 Face.ipynb 的对应关系

| Face.ipynb | 本 notebook |
|---|---|
| `cv.VideoCapture(0)` + `set(3,320) set(4,240) set(5,30)` | `CameraReader`（相同参数，另加后台读帧线程） |
| `imgbox.value = cv.imencode(...)` 回显 | `preview.value = jpg.tobytes()` |
| 线程 + `Arm_serial_servo_write6_array(...)` 驱动舵机 | **不驱动舵机**，只输出 `GestureCommand` |

## 和机械臂联动（C 侧职责）

读取 `gesture_commands.jsonl` 里的 `gesture_code`，查表后经急停/限位校验再下发：

```python
GESTURE_TO_MOTION = {
    1: ("servo", 2, "up", 10),     # FIST
    2: ("servo", 2, "down", 10),   # OPEN_PALM
    3: ("home",),                  # POINT_UP
    7: ("estop",),                 # THUMBS_UP
}
```

## 排错

| 现象 | 原因 | 处理 |
|---|---|---|
| `NameError: HandRecognizer` | 单元 1 没跑成功就跑了后面的单元 | 先跑单元 1，看完整堆栈 |
| 内核自动重启、输出凭空消失 | mediapipe 1.x 的 `rcpc` 非法指令（SIGILL） | 用 `pylibs` 里的 0.10.18；单元 1 会直接拦下 1.x |
| 摄像头能开但读不到画面 | 设备被 Face.ipynb 的后台线程占用 | Shutdown 该内核，或 `fuser -v /dev/video0` 查占用 |
| 模型加载 0.1-3 秒 | mediapipe 0.10.x 加载 20MB 级 `libmediapipe.so` | 正常 |